# Fine-tuning con LoRA de InternVL3.5-8B — Entrada: imagen + texto

## Importar librerías

In [1]:
# Instalar/actualizar dependencias necesarias
%pip install transformers>=4.45.0 peft accelerate bitsandbytes -U -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
from torch import nn
import pandas as pd
import numpy as np
import torch
import json
import os
import gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    roc_curve,
    auc
)

from transformers import (
    AutoModelForImageTextToText, 
    AutoProcessor,
    TrainingArguments, 
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset as TorchDataset

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.metrics.metricfactory import MetricFactory

## Configuración y parámetros

In [3]:
os.environ["HF_TOKEN"] = ""

# ====================
# CONFIGURACIÓN MODELO
# ====================
MODEL_NAME = "OpenGVLab/InternVL3_5-8B-HF"
MAIN_PATH  = ".."
GROUP_ID   = "BeingChillingWeWillWin"
MODEL_ID   = "internvl35_8b_ft"

TEXT_COLUMN  = "combined_text"
LABEL_COLUMN = "label"

# ====================
# RUTAS DE DATOS
# ====================
DATA_TRAIN_PATH = os.path.join(MAIN_PATH, "preprocessed_data", "train_split.json")
DATA_VAL_PATH   = os.path.join(MAIN_PATH, "preprocessed_data", "dev_split.json")
DATA_TEST_PATH  = os.path.join(MAIN_PATH, "preprocessed_data", "test_split.json")

DATA_BASE_DIR   = os.path.join(MAIN_PATH, "materials", "dataset_task2_exist2026")
OUTPUT_DIR      = os.path.join(MAIN_PATH, "weights", f"InternVL35-8B_{TEXT_COLUMN}_lora")
SAVE_PATH       = os.path.join(MAIN_PATH, "weights", f"InternVL35-8B_{TEXT_COLUMN}_final")
PREDICTIONS_DIR = os.path.join(MAIN_PATH, "predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# ====================
# HIPERPARÁMETROS
# ====================
MAX_LENGTH     = 2048  # Aumentado para evitar truncar tokens de imagen
MAX_TEXT_LENGTH = 400  # Truncar el texto de entrada antes del template
MAX_NEW_TOKENS = 10
NUM_EPOCHS     = 3
LEARNING_RATE  = 2e-5
BATCH_SIZE     = 2
GRAD_ACCUM     = 8  # Effective batch size = BATCH_SIZE * GRAD_ACCUM = 16

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

label_map         = {"NO": 0, "YES": 1}
label_map_inverse = {0: "NO", 1: "YES"}

## Carga y preprocesamiento de datos

In [4]:
def load_json_dataset(path):
    """Carga el JSON orientado a diccionario y devuelve un DataFrame."""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data.values())

def build_combined_text(row):
    """Construye la cadena combinada a partir de image_description y text."""
    img_desc = str(row.get('image_description', '') or '')
    txt      = str(row.get('text', '') or '')
    return f"descripcion imagen: {img_desc}. Texto: {txt}"

train_df = load_json_dataset(DATA_TRAIN_PATH)
val_df   = load_json_dataset(DATA_VAL_PATH)
test_df  = load_json_dataset(DATA_TEST_PATH)

for df in [train_df, val_df, test_df]:
    df[TEXT_COLUMN] = df.apply(build_combined_text, axis=1)

train_df["label_int"] = train_df[LABEL_COLUMN].map(label_map)
val_df["label_int"]   = val_df[LABEL_COLUMN].map(label_map)
test_df["label_int"]  = -1  # placeholder para test sin gold

print(f"Text column used : {TEXT_COLUMN}")
print(f"Ejemplo de entrada:\n  {train_df[TEXT_COLUMN].iloc[0][:200]}")
print(f"\nTrain size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"\nDistribución de etiquetas en TRAIN:")
print(train_df[LABEL_COLUMN].value_counts())
print(f"\nDistribución de etiquetas en VAL:")
print(val_df[LABEL_COLUMN].value_counts())

Text column used : combined_text
Ejemplo de entrada:
  descripcion imagen: a close up of a snake with its mouth open and its tongue out. Texto: Demostración de que las cosas mas peligrosas del mundo tienen el mismo aspecto. mémenoides 

Train size: 2146 | Val size: 537 | Test size: 687

Distribución de etiquetas en TRAIN:
label
YES    1282
NO      864
Name: count, dtype: int64

Distribución de etiquetas en VAL:
label
YES    321
NO     216
Name: count, dtype: int64


## Carga del modelo con cuantización y configuración LoRA

In [5]:
# Limpiar memoria antes de cargar
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Configuración de cuantización 8-bit para reducir uso de memoria
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=DTYPE,
)

print(f"Cargando modelo {MODEL_NAME}...")

# Cargar modelo con cuantización
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto",
)

# Cargar processor
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Preparar modelo para entrenamiento con cuantización
model = prepare_model_for_kbit_training(model)

# Configuración LoRA - apuntamos a los módulos de atención del modelo
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Módulos de atención
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Aplicar LoRA al modelo
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n✓ Modelo InternVL3.5-8B-HF cargado con LoRA")
if torch.cuda.is_available():
    print(f"  Memoria GPU usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Cargando modelo OpenGVLab/InternVL3_5-8B-HF...


Loading weights:   0%|          | 0/841 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 17,694,720 || all params: 8,546,013,184 || trainable%: 0.2071

✓ Modelo InternVL3.5-8B-HF cargado con LoRA
  Memoria GPU usada: 12.35 GB


## Dataset y Data Collator para Fine-tuning

In [6]:
class SexismMemeDataset(TorchDataset):
    """Dataset para clasificación de memes con imagen + texto siguiendo formato InternVL."""
    
    def __init__(self, df, processor, base_dir, max_length=2048, max_text_length=400):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.base_dir = Path(base_dir)
        self.max_length = max_length
        self.max_text_length = max_text_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Cargar imagen
        img_path = self.base_dir / row['path_memes']
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"[WARN] Error cargando imagen {img_path}: {e}")
            # Crear imagen placeholder
            image = Image.new('RGB', (224, 224), color='gray')
        
        # Construir prompt
        combined_text = row[TEXT_COLUMN]
        label = row['label_int']
        target_text = "YES" if label == 1 else "NO"
        
        # CRÍTICO: Truncar el texto ANTES de aplicar el template para evitar prompts muy largos
        # que causen que se corten tokens de imagen durante la tokenización
        if len(combined_text) > self.max_text_length:
            combined_text = combined_text[:self.max_text_length] + "..."
        
        # Formato para InternVL con mensajes estructurados
        # InternVL necesita formato de mensajes con tipos (image, text), NO el token <image> manual
        user_content = f"Analyze this meme. {combined_text}\n\nIs this meme sexist? Answer only YES or NO."
        
        # Construir mensajes en formato que apply_chat_template entiende
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},  # Placeholder para imagen - se reemplaza con la imagen real
                    {"type": "text", "text": user_content}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": target_text}]
            }
        ]
        
        # Aplicar chat template para generar el prompt con los placeholders correctos
        try:
            prompt = self.processor.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=False
            )
        except Exception as e:
            print(f"[ERROR] Chat template fallido para idx {idx}: {e}")
            # Fallback: formato simple sin template
            prompt = f"{user_content}\n{target_text}"
        
        # Tokenizar con imagen
        # CRÍTICO: Usar truncation=True para manejar imágenes con resolución dinámica
        # InternVL genera diferente cantidad de tokens por imagen, debemos truncar para consistencia
        try:
            encoding = self.processor(
                text=prompt,
                images=[image],
                padding="max_length",
                max_length=self.max_length,
                truncation=True,  # AÑADIDO: Forzar truncamiento a max_length
                return_tensors="pt"
            )
        except Exception as e:
            print(f"[ERROR] Procesamiento fallido para idx {idx}: {e}")
            print(f"  Prompt length: {len(prompt)} chars")
            # Fallback: acortar más el texto
            short_content = f"Analyze this meme. Is it sexist? Answer YES or NO."
            messages_short = [
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": short_content}]},
                {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
            ]
            try:
                prompt = self.processor.apply_chat_template(messages_short, tokenize=False, add_generation_prompt=False)
            except:
                prompt = f"{short_content}\n{target_text}"
            
            encoding = self.processor(
                text=prompt,
                images=[image],
                padding="max_length",
                max_length=self.max_length,
                truncation=True,  # AÑADIDO: Forzar truncamiento en fallback también
                return_tensors="pt"
            )
        
        # Squeeze para eliminar dimensión batch de PT
        # CUIDADO: No hacer squeeze ciego a pixel_values si tiene estructura especial
        item = {}
        for k, v in encoding.items():
            if isinstance(v, torch.Tensor):
                # Para input_ids y attention_mask, hacer squeeze
                if k in ['input_ids', 'attention_mask']:
                    item[k] = v.squeeze(0)
                else:
                    # Para pixel_values y otros, mantener estructura o hacer squeeze cuidadoso
                    if v.dim() > 1 and v.shape[0] == 1:
                        item[k] = v.squeeze(0)
                    else:
                        item[k] = v
            else:
                item[k] = v
        
        # Crear labels - solo calculamos loss en la respuesta del asistente
        item['labels'] = item['input_ids'].clone()
        
        # IMPORTANTE: Intentar maskear el prompt para calcular loss solo en la respuesta
        # Esto requiere encontrar dónde empieza la respuesta del asistente
        # Por simplicidad, maskeamos todo excepto los últimos tokens (respuesta corta YES/NO)
        # En producción, deberías hacer un masking más preciso basado en el template
        
        return item


class DataCollatorForVLM:
    """Collator robusto para Vision-Language Model (InternVL)."""
    
    def __init__(self, processor, pad_token_id=None):
        self.processor = processor
        self.pad_token_id = pad_token_id or processor.tokenizer.pad_token_id
        
    def __call__(self, features):
        batch = {}
        
        # Handle input_ids - con padding dinámico por si acaso hay diferencias
        if 'input_ids' in features[0]:
            # Verificar si todos tienen el mismo tamaño
            lengths = [f['input_ids'].shape[0] for f in features]
            if len(set(lengths)) > 1:
                # Diferentes longitudes - hacer padding dinámico al máximo del batch
                max_length = max(lengths)
                padded_input_ids = []
                for f in features:
                    pad_length = max_length - f['input_ids'].shape[0]
                    if pad_length > 0:
                        padding = torch.full((pad_length,), self.pad_token_id, dtype=f['input_ids'].dtype)
                        padded_input_ids.append(torch.cat([f['input_ids'], padding]))
                    else:
                        padded_input_ids.append(f['input_ids'])
                batch['input_ids'] = torch.stack(padded_input_ids)
            else:
                # Mismo tamaño - stack directo
                batch['input_ids'] = torch.stack([f['input_ids'] for f in features])
            
        # Handle attention_mask - con padding dinámico por si acaso
        if 'attention_mask' in features[0]:
            lengths = [f['attention_mask'].shape[0] for f in features]
            if len(set(lengths)) > 1:
                max_length = max(lengths)
                padded_masks = []
                for f in features:
                    pad_length = max_length - f['attention_mask'].shape[0]
                    if pad_length > 0:
                        padding = torch.zeros((pad_length,), dtype=f['attention_mask'].dtype)
                        padded_masks.append(torch.cat([f['attention_mask'], padding]))
                    else:
                        padded_masks.append(f['attention_mask'])
                batch['attention_mask'] = torch.stack(padded_masks)
            else:
                batch['attention_mask'] = torch.stack([f['attention_mask'] for f in features])
            
        # Handle pixel_values (imágenes) - apilar correctamente
        if 'pixel_values' in features[0]:
            pixel_values_list = [f['pixel_values'] for f in features]
            # Verificar dimensiones y apilar
            if isinstance(pixel_values_list[0], torch.Tensor):
                if len(pixel_values_list[0].shape) == 3:
                    # Shape: (C, H, W) -> apilar a (B, C, H, W)
                    batch['pixel_values'] = torch.stack(pixel_values_list)
                elif len(pixel_values_list[0].shape) == 4:
                    # Shape: (N, C, H, W) para dynamic patches -> concatenar
                    batch['pixel_values'] = torch.cat(pixel_values_list, dim=0)
                else:
                    batch['pixel_values'] = torch.stack(pixel_values_list)
            else:
                batch['pixel_values'] = pixel_values_list
            
        # Handle labels - con padding dinámico por si acaso
        if 'labels' in features[0]:
            lengths = [f['labels'].shape[0] for f in features]
            if len(set(lengths)) > 1:
                max_length = max(lengths)
                padded_labels = []
                for f in features:
                    pad_length = max_length - f['labels'].shape[0]
                    if pad_length > 0:
                        padding = torch.full((pad_length,), -100, dtype=f['labels'].dtype)
                        padded_labels.append(torch.cat([f['labels'], padding]))
                    else:
                        padded_labels.append(f['labels'])
                batch['labels'] = torch.stack(padded_labels)
            else:
                batch['labels'] = torch.stack([f['labels'] for f in features])
            # Reemplazar pad tokens con -100 para ignorar en loss
            batch['labels'][batch['labels'] == self.pad_token_id] = -100
            
        # Handle image_flags si existe (InternVL específico)
        if 'image_flags' in features[0]:
            batch['image_flags'] = torch.stack([f['image_flags'] for f in features])
            
        # Handle pixel_attention_mask si existe
        if 'pixel_attention_mask' in features[0]:
            batch['pixel_attention_mask'] = torch.stack([f['pixel_attention_mask'] for f in features])
            
        return batch


# Crear datasets
print("Creando datasets...")
train_dataset = SexismMemeDataset(train_df, processor, DATA_BASE_DIR, MAX_LENGTH, MAX_TEXT_LENGTH)
eval_dataset = SexismMemeDataset(val_df, processor, DATA_BASE_DIR, MAX_LENGTH, MAX_TEXT_LENGTH)

print(f"✓ Train dataset: {len(train_dataset)} ejemplos")
print(f"✓ Eval dataset: {len(eval_dataset)} ejemplos")

# Verificar un ejemplo
sample = train_dataset[0]
print(f"\nEjemplo de entrada:")
print(f"  input_ids shape: {sample['input_ids'].shape}")
if 'pixel_values' in sample:
    print(f"  pixel_values shape: {sample['pixel_values'].shape}")
print(f"  labels shape: {sample['labels'].shape}")

# Verificar que el prompt tiene el formato correcto
decoded_text = processor.tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"\nTexto decodificado (primeros 500 chars):")
print(f"  {decoded_text[:500]}...")

# Contar tokens reales (sin padding)
non_pad_tokens = (sample['input_ids'] != processor.tokenizer.pad_token_id).sum().item()
print(f"\nEstadísticas del prompt:")
print(f"  Longitud total: {len(sample['input_ids'])}")
print(f"  Tokens reales (sin padding): {non_pad_tokens}")
print(f"  Tokens de padding: {len(sample['input_ids']) - non_pad_tokens}")

# Verificar tokens especiales de InternVL (pueden variar según la versión)
special_tokens = ['<image>', '[IMG]', '<img>', 'Image:', '<IMG>', 'IMG_CONTEXT']
found_tokens = [token for token in special_tokens if token in decoded_text]
if found_tokens:
    print(f"✓ Tokens de imagen encontrados: {found_tokens}")
else:
    print("⚠ ADVERTENCIA: No se encontraron tokens de imagen estándar")
    print(f"  Esto puede ser normal si InternVL usa tokens especiales no legibles")

Creando datasets...
✓ Train dataset: 2146 ejemplos
✓ Eval dataset: 537 ejemplos
[ERROR] Procesamiento fallido para idx 0: Mismatch in `image` token count between text and `input_ids`. Got ids=[2044] and text=[3328]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`.
  Prompt length: 322 chars


ValueError: Mismatch in `image` token count between text and `input_ids`. Got ids=[2044] and text=[3328]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`.

## Configuración del Trainer y métricas

In [ ]:
# Data collator
data_collator = DataCollatorForVLM(processor)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,  # Importante para mantener pixel_values
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Mejor compatibilidad
    optim="paged_adamw_8bit",  # Optimizador que ahorra memoria
    max_grad_norm=1.0,  # Gradient clipping para estabilidad
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer configurado")

## Entrenamiento

In [ ]:
# Ejecutar entrenamiento
print("Iniciando entrenamiento...")
trainer.train()

# Guardar modelo final
print("\nGuardando modelo...")
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"✓ Modelo guardado en: {SAVE_PATH}")

## Inferencia en DEV y cálculo de métricas

In [ ]:
@torch.no_grad()
def generate_prediction(row, model, processor, base_dir):
    """Genera predicción para una muestra usando formato InternVL con mensajes estructurados."""
    img_path = Path(base_dir) / row['path_memes']
    combined_text = row[TEXT_COLUMN]
    
    try:
        image = Image.open(img_path).convert('RGB')
    except:
        return {'classification': 'NO', 'confidence': 0.0}
    
    # Truncar el texto para evitar prompts demasiado largos
    if len(combined_text) > MAX_TEXT_LENGTH:
        combined_text = combined_text[:MAX_TEXT_LENGTH] + "..."
    
    # Construir mensajes en formato estructurado que InternVL espera
    user_content = f"Analyze this meme. {combined_text}\n\nIs this meme sexist? Answer only YES or NO."
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # Placeholder para imagen
                {"type": "text", "text": user_content}
            ]
        }
    ]
    
    # Aplicar chat template con generation prompt
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    
    # Procesar inputs - sin truncation para no cortar tokens de imagen
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(model.device)
    
    # Generar respuesta
    outputs = model.generate(
        **inputs, 
        max_new_tokens=MAX_NEW_TOKENS, 
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id
    )
    
    # Decodificar solo los tokens generados (no el prompt)
    generated_ids = outputs[:, inputs['input_ids'].shape[1]:]
    response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip().upper()
    
    # Parsear respuesta
    if "YES" in response:
        return {'classification': 'YES', 'confidence': 0.9}
    else:
        return {'classification': 'NO', 'confidence': 0.9}


def save_probs_json(ids, probs, split_name, labels=None):
    records = []
    for i, (id_exist, prob) in enumerate(zip(ids, probs)):
        rec = {'id': str(id_exist), 'prob_YES': round(float(prob), 6)}
        if labels is not None:
            rec['label'] = label_map_inverse[int(labels[i])]
        records.append(rec)
    path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_{split_name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


# Evaluar en DEV
model.eval()
print("Evaluando en DEV...")

val_results = []
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Inferencia DEV"):
    pred = generate_prediction(row, model, processor, DATA_BASE_DIR)
    val_results.append({
        'id_EXIST': str(row['id_EXIST']),
        'classification': pred['classification'],
        'confidence': pred['confidence'],
        'true_label': label_map_inverse[row['label_int']]
    })

y_pred_labels = [label_map[r['classification']] for r in val_results]
y_true_labels = [label_map[r['true_label']] for r in val_results]
y_probs_dev = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in val_results]

# Métricas
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_labels, y_pred_labels, average='binary', zero_division=0
)

save_probs_json(val_df['id_EXIST'].values, y_probs_dev, 'dev', labels=val_df['label_int'].values)

print(f"\n=== Métricas en DEV (Fine-tuned) ===")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall   : {recall:.4f}")
print(f"  F1-Score : {f1:.4f}")

# ROC AUC
fpr, tpr, _ = roc_curve(y_true_labels, y_probs_dev)
roc_auc = auc(fpr, tpr)
print(f"\nAUC (DEV): {roc_auc:.4f}")

# Gráfica ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve DEV — InternVL3.5-8B Fine-tuned')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluación en DEV con PyEvALL

In [ ]:
# Preparar datos para PyEvALL
dev_preds_for_pyevall = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in val_results
]
dev_preds_df = pd.DataFrame(dev_preds_for_pyevall)
dev_preds_path = os.path.join(PREDICTIONS_DIR, 'dev_predictions_temp.json')
with open(dev_preds_path, 'w', encoding='utf-8') as f:
    f.write(dev_preds_df.to_json(orient='records'))

dev_gold = [
    {'test_case': 'EXIST2025', 'id': str(id_exist), 'value': label}
    for id_exist, label in zip(val_df['id_EXIST'].values, val_df[LABEL_COLUMN].values)
]
dev_gold_df = pd.DataFrame(dev_gold)
dev_gold_path = os.path.join(PREDICTIONS_DIR, 'dev_gold_temp.json')
with open(dev_gold_path, 'w', encoding='utf-8') as f:
    f.write(dev_gold_df.to_json(orient='records'))

# Evaluar con PyEvALL
test_eval = PyEvALLEvaluation()
metrics = [MetricFactory.Accuracy.value, MetricFactory.FMeasure.value]
report = test_eval.evaluate(dev_preds_path, dev_gold_path, metrics)
print("\n=== Evaluación en DEV con PyEvALL ===")
report.print_report()

## Inferencia en TEST y generación de predicciones finales

In [ ]:
# Inferencia en TEST
print("Evaluando en TEST...")

test_results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia TEST"):
    pred = generate_prediction(row, model, processor, DATA_BASE_DIR)
    test_results.append({
        'id_EXIST': str(row['id_EXIST']),
        'classification': pred['classification'],
        'confidence': pred['confidence']
    })

y_probs_test = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in test_results]
test_preds = [r['classification'] for r in test_results]

save_probs_json(test_df['id_EXIST'].values, y_probs_test, 'test')

print(f"\nPredicciones en TEST:")
print(f"  Total: {len(test_preds)}")
print(f"  YES  : {sum(1 for p in test_preds if p == 'YES')} ({100*sum(1 for p in test_preds if p == 'YES')/len(test_preds):.2f}%)")
print(f"  NO   : {sum(1 for p in test_preds if p == 'NO')} ({100*sum(1 for p in test_preds if p == 'NO')/len(test_preds):.2f}%)")

## Guardar predicciones en formato PyEvALL para TEST

In [ ]:
# Guardar predicciones en formato PyEvALL
test_preds_for_submission = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in test_results
]
test_preds_df = pd.DataFrame(test_preds_for_submission)

output_filename = f"{GROUP_ID}_{MODEL_ID}.json"
output_path = os.path.join(PREDICTIONS_DIR, output_filename)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(test_preds_df.to_json(orient='records'))

print(f"\n✓ Predicciones guardadas en: {output_path}")